## fmuritalKNN.ipynb
## Supervised Learning: Naïve Bayesian Classifier using K-fold Cross Validation (K=2)
## Use Case: Sentiment Analysis of Movie Reviews
## Faruk Muritala
## Oct 23 2023

In [ ]:
# Section 1: Import the 'getpass' function from the 'getpass' module to securely input sensitive information like API tokens

from getpass import getpass


In [ ]:
# Section 2: Import OS and Install Necessary Packages

# Import the 'os' module to interact with the operating system
import os

# Import the 'numpy' library for numerical operations
import numpy as np

# Install necessary packages using pip
# - 'langchain' for language processing
# - 'sentence_transformers' for working with sentence embeddings
# - 'chromadb' for working with vector stores
# - 'transformers', 'sentencepiece', and 'sacremoses' for language model support
!pip install langchain
!pip install sentence_transformers
!pip install chromadb
!pip install transformers sentencepiece sacremoses
!pip install -q transformers
!pip install -q simpletransformers
!pip install -q datasets



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 13.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.4/49.4 kB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.0/86.0 kB 957.1 kB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 39.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 46.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.0/302.0 kB 24.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 78.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 59.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.0/295.0 kB 26.5 MB/s eta 0:00:00
  Created wheel for sentence_transformers: filename=sentence_transformers-2.2.2-py3-none-any.whl size=125923 sha256=3dad7add70b248f4cff528aafbce18cfb2d86942

In [ ]:
# Section 3: Import Libraries and Modules for Text Processing and Analysis

# Import necessary modules and classes for text processing and analysis
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.document_loaders import WebBaseLoader
from langchain import HuggingFaceHub
from langchain import PromptTemplate, LLMChain
from langchain.indexes import VectorstoreIndexCreator
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma
from collections import Counter
from sklearn.metrics import accuracy_score, confusion_matrix


In [ ]:
# Section 4: Set Up API Token

# Prompt the user to securely input their Hugging Face API token using 'getpass'
HUGGINGFACEHUG_API_TOKEN = getpass()

# Set the API token in the environment variable for later use
os.environ["HUGGINGFACEHUG_API_TOKEN"] = HUGGINGFACEHUG_API_TOKEN


··········


In [ ]:
# Install the "datasets" Python package
#!pip install datasets

# Import NumPy and alias it as "np"
import numpy as np
# Import Pandas and alias it as "pd"
import pandas as pd
# Import the "load_dataset" function from the "datasets" package
from datasets import load_dataset

# Load IMDb training data
dataset_train = load_dataset('imdb', split='train')
# Rename the 'label' column to 'labels' in training data
dataset_train = dataset_train.rename_column('label', 'labels')
# Convert training data to a Pandas DataFrame
train_df = pd.DataFrame(dataset_train)

# Load IMDb testing data
dataset_test = load_dataset('imdb', split='test')
# Rename the 'label' column to 'labels' in testing data
dataset_test = dataset_test.rename_column('label', 'labels')

# Convert testing data to a Pandas DataFrame
test_df = pd.DataFrame(dataset_test)


Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [ ]:
# Create a list containing the training and testing DataFrames
dfs = [train_df, test_df]

# Concatenate (combine) the DataFrames vertically to create a single dataset
total_dataset = pd.concat(dfs)

# Display the first few rows of the combined dataset
total_dataset.head()

# Calculate and display the length (number of rows) of the combined dataset
len(total_dataset)



50000

In [ ]:
# Import the DataFrameLoader class from the "langchain.document_loaders" module
from langchain.document_loaders import DataFrameLoader

# Create an instance of the DataFrameLoader class with the combined dataset
# Specify the 'text' column as the page content column
loader = DataFrameLoader(total_dataset, page_content_column='text')

# Load data using the DataFrameLoader instance and store it in the 'data' variable
data = loader.load()


In [ ]:
Practice_data = data[::2][:25000]
#FOR PRACTICE I USE 25000 TO SET THE ALGORITHM IN A SYSTEMATIC WAY (TAKING 2nd POINT) and name it Practice data
Practice_data

[Document(page_content='I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are 

In [ ]:
# Section 7: Vectorstore Creation

# Step 5: Vectorstore Creation
# Note: If using the OpenAIEmbeddings, change embedding to OpenAIEmbeddings in the Vector store
# from langchain.embeddings import OpenAIEmbeddings

# Create a list of IDs for the documents

#ids = [str(i) for i in range(1, len(Practice_data) + 1)]

ids = [str(i + 1) for i in range(len(Practice_data))]


# Create a Chroma vector store from the preprocessed text segments using HuggingFaceEmbeddings
vectorstore = Chroma.from_documents(documents=Practice_data, embedding=HuggingFaceEmbeddings(), ids=ids)


In [ ]:
# Section 8: Checking Individual IDs and Result

# Checking for individual IDs
# You can uncomment and use this line to check a specific ID's embeddings, documents, and metadata.
# print(vectorstore._collection.get(ids=[ids[20]], include=['embeddings', 'documents', 'metadatas']))

# Retrieving all results including embeddings, documents, and metadata
result = vectorstore._collection.get(include=['embeddings', 'documents', 'metadatas'])
#result = vectorstore

In [ ]:
#result["metadatas"]

In [ ]:
# Create a DataFrame containing embeddings and metadatas from the vectorstore
embframe = pd.DataFrame(vectorstore._collection.get(include=["embeddings", "metadatas"]))

# Extract the 'labels' from the 'metadatas' in embframe and store them in a list
labels = [x['labels'] for x in embframe['metadatas']]

# Create a DataFrame from the extracted 'labels'
labeling = pd.DataFrame(labels)


In [ ]:
# Add a new column "labeling" to the "embframe" DataFrame and assign the "labeling" values to it
embframe["labeling"] = labeling

# Display the updated "embframe" DataFrame
embframe


,ids,embeddings,metadatas,documents,labeling
0,1,"[0.051543161273002625, 0.0489380918443203, -0....",{'labels': 0},None,0
1,2,"[-0.022716065868735313, 0.08454512059688568, 0...",{'labels': 0},None,0
2,3,"[0.004128900822252035, 0.08748075366020203, 0....",{'labels': 0},None,0
3,4,"[0.033928193151950836, 0.1193242222070694, -0....",{'labels': 0},None,0
4,5,"[0.007491306867450476, 0.01051307562738657, -0...",{'labels': 0},None,0
...,...,...,...,...,...
24995,24996,"[0.055829111486673355, -0.034562356770038605, ...",{'labels': 1},None,1
24996,24997,"[0.023964637890458107, -0.007764998357743025, ...",{'labels': 1},None,1
24997,24998,"[0.037451524287462234, 0.005957203917205334, -...",{'labels': 1},None,1
24998,24999,"[0.03981273993849754, -0.009968959726393223, 0...",{'labels': 1},None,1


In [ ]:
# Splitting into 2 folds
from sklearn.model_selection import train_test_split

# Assuming you have your data and target variables named 'X' and 'y'
# X is your data, and y is your target (labels or values to predict)
X = embframe["embeddings"]
y = embframe["labeling"]

# Split the data into two halves for 2-fold cross-validation
train_size = 0.5  # Specified the training set size (50% in each case)


X_fold1, X_fold2, y_fold1, y_fold2 = train_test_split(X, y, test_size=(1 - train_size), random_state=1234)
# Implementing the Naïve Bayesian Classifier for each fold here
# For fold 1, X_fold1 is the training data and X_fold2 is the test data
# Alternatively, for this project:
# For fold 2, X_fold2 is the training data and X_fold1 is the test data



In [ ]:
# Assuming X&y_train and X&y_test are lists of data points

#np.array(list(result['embeddings']))
Practice_data = np.array(Practice_data)
# Convert the 'train' and 'test' list of data points into a NumPy array

X_fold1 = np.array(list(X_fold1))
y_fold1 = np.array(list(y_fold1))

X_fold2 = np.array(list(X_fold2))
y_fold2 = np.array(list(y_fold2))


In [ ]:
import numpy as np

class NaiveBayesianClassifier:
    def fit(self, X, y):
        # Get the number of samples and features in the input data
        n_samples, n_features = X.shape

        # Find unique classes in the target labels
        self.classes = np.unique(y)
        n_classes = len(self.classes)

        # Calculate prior probabilities for each class
        self.priors = np.zeros(n_classes, dtype=np.float64)
        for idx, c in enumerate(self.classes):
            self.priors[idx] = np.sum(y == c) / n_samples

        # Initialize dictionaries to store class-conditional statistics (mean and variance)
        self.mean = {}
        self.var = {}

        # Calculate mean and variance for each feature and class
        for c in self.classes:
            # Get the subset of input data corresponding to the current class
            X_c = X[y == c]

            # Calculate and store the mean for each feature for the current class
            self.mean[c] = X_c.mean(axis=0)

            # Calculate and store the variance for each feature for the current class
            self.var[c] = X_c.var(axis=0)

    def predict(self, X):
        # Predict the class labels for the input data X
        y_pred = [self._predict(x) for x in X]
        return np.array(y_pred)

    def _predict(self, x):
        posteriors = []

        # Calculate posterior probability for each class
        for c in self.classes:
            # Calculate the log of prior probability for the current class
            prior = np.log(self.priors[c])

            # Calculate the sum of log-likelihoods for the current feature vector x
            likelihood = np.sum(np.log(self._pdf(c, x)))

            # Calculate the posterior probability by adding the log-prior and log-likelihood
            posterior = prior + likelihood
            posteriors.append(posterior)

        # Return the class with the highest posterior probability as the predicted class
        return self.classes[np.argmax(posteriors)]

    def _pdf(self, c, x):
        # Calculate the probability density function (pdf) for feature vector x for class c
        mean = self.mean[c]
        var = self.var[c]

        # Calculate the numerator of the pdf
        numerator = np.exp(-((x - mean) ** 2) / (2 * var))

        # Calculate the denominator of the pdf
        denominator = np.sqrt(2 * np.pi * var)

        # Return the pdf as the ratio of the numerator and denominator
        return numerator / denominator


Quuestion 2. Provide a combined confusion matrix and the accuracy for each fold as well as the
complete set by merging results from both folds

In [ ]:
#Predicting fold2 using ford1 model

# Creating an instance of the NaiveBayesianClassifier for fold 1
nb_classifier_fold1 = NaiveBayesianClassifier()

# Train the classifier on the training data for fold 1
nb_classifier_fold1.fit(X_fold1, y_fold1)

# Predict the for fold 2 using ford 1 model
predictions_fold1 = nb_classifier_fold1.predict(X_fold2)

# Calculate accuracy and confusion matrix for the predicted fold 2 using ford 1 model
accuracy_fold1 = accuracy_score(y_fold2, predictions_fold1)
conf_matrix_fold1 = confusion_matrix(y_fold2, predictions_fold1)

print("Naive Bayesian Classifier Accuracy (Fold 1):", accuracy_fold1)
print("Confusion Matrix (Fold 1):\n", conf_matrix_fold1)


Naive Bayesian Classifier Accuracy (Fold 1): 0.82072
Confusion Matrix (Fold 1):
 [[4892 1341]
 [ 900 5367]]


In [ ]:
#Predicting fold 1 using ford2 model

# Creating an instance of the NaiveBayesianClassifier for fold 2
nb_classifier_fold2 = NaiveBayesianClassifier()

# Train the classifier on the training data for fold 2
nb_classifier_fold2.fit(X_fold2, y_fold2)

# Predict the for fold 1 using ford 2 model
predictions_fold2 = nb_classifier_fold2.predict(X_fold1)

# Calculate accuracy and confusion matrix for the Predicted fold 1 using ford 2 model
accuracy_fold2 = accuracy_score(y_fold1, predictions_fold2)
conf_matrix_fold2 = confusion_matrix(y_fold1, predictions_fold2)

print("Naive Bayesian Classifier Accuracy (Fold 2):", accuracy_fold2)
print("Confusion Matrix (Fold 2):\n", conf_matrix_fold2)


Naive Bayesian Classifier Accuracy (Fold 2): 0.81936
Confusion Matrix (Fold 2):
 [[4942 1325]
 [ 933 5300]]


In [ ]:
#Quuestion 2b. Providing a combined confusion matrix merging results from both folds

from sklearn.metrics import accuracy_score, confusion_matrix

# Combine predictions from both folds
combined_predictions = np.concatenate([predictions_fold1, predictions_fold2])

# Combine true labels (ground truth) from both folds
combined_true_labels = np.concatenate([y_fold1, y_fold2])

# Calculate combined accuracy
combined_accuracy = accuracy_score(combined_true_labels, combined_predictions)

# Calculate combined confusion matrix
combined_confusion_matrix = confusion_matrix(combined_true_labels, combined_predictions)

print("Combined Naive Bayesian Classifier Accuracy:", combined_accuracy)
print("Combined Confusion Matrix:\n", combined_confusion_matrix)


Combined Naive Bayesian Classifier Accuracy: 0.50076
Combined Confusion Matrix:
 [[5843 6657]
 [5824 6676]]


 Quuestion 3: Provide 3 examples of correct classification as positive, 3 examples of correct
classification as negative, 3 examples of incorrect classification as negative, and 3
examples of incorrect classification as positive.

In [ ]:
#Examples of Correct Classification as Positive

correct_positive_indices = np.where((combined_predictions == 1) & (combined_true_labels == 1))[0]

print("Examples of Correct Classification as Positive:")
for idx in correct_positive_indices[:3]:
    predicted_label = combined_predictions[idx]
    actual_label = combined_true_labels[idx]
    text_review = Practice_data[idx]  # Access the text review from Practice_data based on the index
    print(f"Predicted: {predicted_label} | Actual: {actual_label}")
    print("Review Text:", text_review)



Examples of Correct Classification as Positive:
Predicted: 1 | Actual: 1
Review Text: page_content='It was great to see some of my favorite stars of 30 years ago including John Ritter, Ben Gazarra and Audrey Hepburn. They looked quite wonderful. But that was it. They were not given any characters or good lines to work with. I neither understood or cared what the characters were doing.<br /><br />Some of the smaller female roles were fine, Patty Henson and Colleen Camp were quite competent and confident in their small sidekick parts. They showed some talent and it is sad they didn\'t go on to star in more and better films. Sadly, I didn\'t think Dorothy Stratten got a chance to act in this her only important film role.<br /><br />The film appears to have some fans, and I was very open-minded when I started watching it. I am a big Peter Bogdanovich fan and I enjoyed his last movie, "Cat\'s Meow" and all his early ones from "Targets" to "Nickleodeon". So, it really surprised me that I was

In [ ]:

#Examples of Correct Classification as Negative
correct_negative_indices = np.where((combined_predictions == 0) & (combined_true_labels == 0))[0]

print("Examples of Correct Classification as Negative:")
for idx in correct_negative_indices[:3]:
    predicted_label = combined_predictions[idx]
    actual_label = combined_true_labels[idx]
    text_review = Practice_data[idx]  # Access the text review from Practice_data based on the index
    print(f"Predicted: {predicted_label} | Actual: {actual_label}")
    print("Review Text:", text_review)



Examples of Correct Classification as Negative:
Predicted: 0 | Actual: 0
Review Text: page_content='Oh, brother...after hearing about this ridiculous film for umpteen years all I can think of is that old Peggy Lee song..<br /><br />"Is that all there is??" ...I was just an early teen when this smoked fish hit the U.S. I was too young to get in the theater (although I did manage to sneak into "Goodbye Columbus"). Then a screening at a local film museum beckoned - Finally I could see this film, except now I was as old as my parents were when they schlepped to see it!!<br /><br />The ONLY reason this film was not condemned to the anonymous sands of time was because of the obscenity case sparked by its U.S. release. MILLIONS of people flocked to this stinker, thinking they were going to see a sex film...Instead, they got lots of closeups of gnarly, repulsive Swedes, on-street interviews in bland shopping malls, asinie political pretension...and feeble who-cares simulated sex scenes with sa

In [ ]:
#Examples of Incorrect Classification as Negative

incorrect_negative_indices = np.where((combined_predictions == 0) & (combined_true_labels == 1))[0]

print("Examples of Incorrect Classification as Negative:")
for idx in incorrect_negative_indices[:3]:
    predicted_label = combined_predictions[idx]
    actual_label = combined_true_labels[idx]
    text_review = Practice_data[idx]  # Access the text review from Practice_data based on the index
    print(f"Predicted: {predicted_label} | Actual: {actual_label}")
    print("Review Text:", text_review)


Examples of Incorrect Classification as Negative:
Predicted: 0 | Actual: 1
Review Text: page_content='I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years

In [ ]:
#Examples of Incorrect Classification as Positive
incorrect_positive_indices = np.where((combined_predictions == 1) & (combined_true_labels == 0))[0]

print("Examples of Incorrect Classification as Positive:")
for idx in incorrect_positive_indices[:3]:
    predicted_label = combined_predictions[idx]
    actual_label = combined_true_labels[idx]
    text_review = Practice_data[idx]  # Access the text review from Practice_data based on the index
    print(f"Predicted: {predicted_label} | Actual: {actual_label}")
    print("Review Text:", text_review)

Examples of Incorrect Classification as Positive:
Predicted: 1 | Actual: 0
Review Text: page_content="If only to avoid making this type of film in the future. This film is interesting as an experiment but tells no cogent story.<br /><br />One might feel virtuous for sitting thru it because it touches on so many IMPORTANT issues but it does so without any discernable motive. The viewer comes away with no new perspectives (unless one comes up with one while one's mind wanders, as it will invariably do during this pointless film).<br /><br />One might better spend one's time staring out a window at a tree growing.<br /><br />" metadata={'labels': 0}
Predicted: 1 | Actual: 0
Review Text: page_content="My interest in Dorothy Stratten caused me to purchase this video. Although it had great actors/actresses, there were just too many subplots going on to retain interest. Plus it just wasn't that interesting. Dialogue was stiff and confusing and the story just flipped around too much to be beli

 5. For the first 5 attributes (embeddings), provide the mean and standard deviation for
each class. Then provide the probability of each class. Note: You need to generate
all this information for each training(fold).

In [ ]:
#For fold1

#Calculate Mean and Standard Deviation for Each Class and Feature
# Calculate mean and standard deviation for each class and feature
mean_by_class = {}
std_by_class = {}

for c in np.unique(y_fold1):
    mean_by_class[c] = X_fold1[y_fold1 == c].mean(axis=0)
    std_by_class[c] = X_fold1[y_fold1 == c].std(axis=0)

# Display the mean and standard deviation for the first 5 features
for feature in range(5):  # Adjust number (5) as needed
    print(f"Feature {feature + 1}:")
    for c in np.unique(y_fold1):
        print(f"Class {c} - Mean: {mean_by_class[c][feature]}, Std Dev: {std_by_class[c][feature]}")


Feature 1:
Class 0 - Mean: 0.012126424534817276, Std Dev: 0.028738343384214132
Class 1 - Mean: -0.0021198450456205147, Std Dev: 0.030444588803384565
Feature 2:
Class 0 - Mean: 0.036666661534417375, Std Dev: 0.03355692475188518
Class 1 - Mean: 0.024567484458889205, Std Dev: 0.03432639326048021
Feature 3:
Class 0 - Mean: 0.00694484020145221, Std Dev: 0.01642604130746843
Class 1 - Mean: 0.004622772375835189, Std Dev: 0.01637636205097766
Feature 4:
Class 0 - Mean: 0.007128764940070408, Std Dev: 0.026229208550030372
Class 1 - Mean: 0.007277890815121752, Std Dev: 0.026249763594690233
Feature 5:
Class 0 - Mean: -0.018424796896295396, Std Dev: 0.028330044950707997
Class 1 - Mean: -0.0064967770427422624, Std Dev: 0.029087330989997752


In [ ]:

# Providing the probability of each class of fold1
class_probabilities = {}

for c in np.unique(y_fold1):
    class_probabilities[c] = np.sum(y_fold1 == c) / len(y_fold1)

# Display class probabilities
for c, probability in class_probabilities.items():
    print(f"Class {c} Probability: {probability}")


Class 0 Probability: 0.50136
Class 1 Probability: 0.49864


In [ ]:
#Now, fold2

#Calculate Mean and Standard Deviation for Each Class and Feature
# Calculate mean and standard deviation for each class and feature
mean_by_class = {}
std_by_class = {}

for c in np.unique(y_fold2):
    mean_by_class[c] = X_fold2[y_fold2 == c].mean(axis=0)
    std_by_class[c] = X_fold2[y_fold2 == c].std(axis=0)

# Display the mean and standard deviation for the first 5 features
for feature in range(5):  # Adjust number (5) as needed
    print(f"Feature {feature + 1}:")
    for c in np.unique(y_fold2):
        print(f"Class {c} - Mean: {mean_by_class[c][feature]}, Std Dev: {std_by_class[c][feature]}")


Feature 1:
Class 0 - Mean: 0.012037695505468782, Std Dev: 0.028433128595068316
Class 1 - Mean: -0.001568178436295234, Std Dev: 0.030851602628094207
Feature 2:
Class 0 - Mean: 0.03660667837737691, Std Dev: 0.033397242769834544
Class 1 - Mean: 0.024274725850440913, Std Dev: 0.03447447836545192
Feature 3:
Class 0 - Mean: 0.007447123276414974, Std Dev: 0.016165232522036478
Class 1 - Mean: 0.004728137052146353, Std Dev: 0.016480225484641387
Feature 4:
Class 0 - Mean: 0.0072486520087083, Std Dev: 0.025819737365892596
Class 1 - Mean: 0.007865843276592363, Std Dev: 0.026605804737840584
Feature 5:
Class 0 - Mean: -0.017705912997725042, Std Dev: 0.028242747928994047
Class 1 - Mean: -0.006801438266475048, Std Dev: 0.029250057797755576


In [ ]:

# Providing the probability of each class of fold2
class_probabilities = {}

for c in np.unique(y_fold2):
    class_probabilities[c] = np.sum(y_fold2 == c) / len(y_fold2)

# Display class probabilities
for c, probability in class_probabilities.items():
    print(f"Class {c} Probability: {probability}")


Class 0 Probability: 0.49864
Class 1 Probability: 0.50136
